WARNING: Notebook was written during development. May not work as intended. Retained here only for completion and future reference.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import scipy.optimize as sopt

import crops.src.bc as _bc
import crops.src.contact as _contact
import crops.src.fe_cosserat as _fe
import crops.src.mesher as _mesher
import crops.src.solver as _solver
import crops.src.utils as _utils
import crops.src.viz as _viz

jax.config.update("jax_enable_x64", True)

In [ ]:
gas_constant = 8.3145
bath_temperature = 298.0  # [K]
molar_volume = 1.8e-5  # water [m^3/mol]

ref_chem_potential = 0.0
dry_chem_potential = -1e5  # Start dry
bath_chem_potential = -550.0  # ~80% Relative Humidity

solvent = _fe.FlorySolvent(
  ref_chem_potential=ref_chem_potential,
  bath_temperature=bath_temperature,
  molar_volume=molar_volume,
)


def free_swelling_stretch(flory_param, shear_modulus, chem_potential):
  """Isotropic free-swelling stretch of an unconstrained fiber.

  With no axial load the fiber swells isotropically, so lambda_t = lambda and
  lambda^3 = 1/phi = J. Substituting into the Flory-Rehner equilibrium gives a
  single scalar equation in phi.
  """
  gas_const_temp = gas_constant * bath_temperature

  def residual(phi):
    return (
      (ref_chem_potential - chem_potential) / gas_const_temp
      + np.log(1.0 - phi)
      + phi
      + flory_param * phi**2
      + (molar_volume * shear_modulus / gas_const_temp) * (phi ** (1.0 / 3.0) - phi)
    )

  phi = sopt.brentq(residual, 1e-9, 1.0 - 1e-9)
  return phi ** (-1.0 / 3.0), phi

In [ ]:
shear_mod = 1.0e6  # same for gel and elastomer
flory_grid = np.linspace(0.05, 0.9, 200)
stretches = np.array(
  [free_swelling_stretch(x, shear_mod, bath_chem_potential)[0] for x in flory_grid]
)

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(flory_grid, stretches)
ax.set_xlabel(r"Flory parameter $\chi$")
ax.set_ylabel(r"free-swelling stretch $\lambda_{free}$")
plt.show()

### Geom

In [ ]:
rod_length = 1.0
num_elems = 100
num_nodes = num_elems + 1
fiber_offset = 0.05  # c [m]
fiber_radius = 0.03

flory_gel = 0.5  # the layer that swells
flory_elastomer = 2.0  # the layer that barely swells

# Offsets along d1 only, so the rod bends about d2 and curls in the x-y plane.
offsets_local = np.array([[fiber_offset, 0.0], [-fiber_offset, 0.0]])
num_fibers = 2
area_per_fiber = np.pi * fiber_radius**2

rod = _mesher.build_rod_geometry(
  analytical_curve=lambda k: jnp.array([k * rod_length, 0.0, 0.0]),
  parametric_coords=jnp.linspace(0.0, 1.0, num_nodes),
  fiber_offsets=jnp.asarray(np.tile(offsets_local, (num_elems, 1, 1))),
  fiber_areas=jnp.full((num_elems, num_fibers), area_per_fiber),
  fiber_mass_density=jnp.ones((num_elems, num_fibers)),
  fiber_centroid_moi=jnp.zeros((num_elems, num_fibers, 2)),
  seed_binormal=jnp.array([0.0, 0.0, 1.0]),
)


In [ ]:
lam_g, phi_g = free_swelling_stretch(flory_gel, shear_mod, bath_chem_potential)
lam_e, phi_e = free_swelling_stretch(flory_elastomer, shear_mod, bath_chem_potential)

nu_exact = 0.5 * (lam_g + lam_e)
curvature_exact = (lam_e - lam_g) / (2.0 * fiber_offset)
bend_angle = curvature_exact * rod_length
arc_radius = nu_exact / curvature_exact

print(f"lambda_g = {lam_g:.6f}  (phi = {phi_g:.6f})")
print(f"lambda_e = {lam_e:.6f}  (phi = {phi_e:.6f})")
print(f"nu       = {nu_exact:.6f}")
print(f"kappa_2  = {curvature_exact:.6f} 1/m")
print(f"total bend angle = {bend_angle:.4f} rad = {np.degrees(bend_angle):.1f} deg")
print(f"arc radius       = {arc_radius:.6f} m")
assert abs(bend_angle) < np.pi, "Bend angle too large, you will hit the 2pi blocker."

### BC

In [ ]:
num_steps = 20
load_factor = jnp.linspace(1.0/num_steps, 1, num_steps)**0.05
chem_potentials = chem_potential = (
        load_factor * bath_chem_potential + (1 - load_factor) * dry_chem_potential
      )

# Clamp only node 0. The rod is otherwise free to swell and curl.
clamped_dofs = (
  _utils.Translation.X,
  _utils.Translation.Y,
  _utils.Translation.Z,
  _utils.Rotation.THETA_X,
  _utils.Rotation.THETA_Y,
  _utils.Rotation.THETA_Z,
)
clamp = [
  _bc.DirichletBC(
    values=(dof, [np.array([0.0]) for _ in range(num_steps)]),
    node_targets=[np.array([0]) for _ in range(num_steps)],
    name=f"clamp_{dof}",
  )
  for dof in clamped_dofs
]

planar = [
  _bc.DirichletBC(
    values=(dof, [np.zeros(num_nodes) for _ in range(num_steps)]),
    node_targets=[np.arange(num_nodes) for _ in range(num_steps)],
    name=f"planar_{dof}",
  )
  for dof in (_utils.Translation.Z, _utils.Rotation.THETA_Y)
]

boundary_conditions = _bc.process_boundary_conditions(rod, clamp + planar)

plt.plot(chem_potentials,  "r*--")

### Solver

In [ ]:
nr_solver = _solver.NewtonRaphson(
  solver_type=_solver.NonlinearSolvers.MODIFIED_NEWTON_RAPHSON,
  nonlinear_settings={
    "max_iter": 300,
    "threshold": 1e-8,
    "abs_threshold": 1e-8,
    "lam_min": 0.01,
    "line_search_max_iter": 15,
    "line_search_shrink": 0.5,
    "line_search_alpha_min": 1e-6,
    "line_search_armijo_c": 1e-4,
  },
  linear_settings={"solver": _solver.LinearSolvers.SCIPY_SPARSE, "rtol": 1e-10},
)
problem = _fe.CosseratProblem(
  rod=rod,
  bc=boundary_conditions,
  nr_solver=nr_solver,
  solvent=solvent,
  bisection_settings={
    "lower": 1e-6,
    "upper": 1.0 - 1e-6,
    "tol": 1e-8,
    "max_iter": 200,
  },
)

flory_parameters = jnp.asarray(
  np.tile(np.array([flory_elastomer, flory_gel]), (num_elems, 1))
)

In [ ]:
u_star, u_history = problem.solve(
  u0=jnp.zeros(rod.num_dofs),
  shear_moduli=jnp.full((num_elems, num_fibers), shear_mod),
  flory_parameters=flory_parameters,
  chem_potentials=chem_potentials,
)

### Compare analytical

In [ ]:
u_nodal = np.asarray(u_star).reshape(num_nodes, rod.num_dofs_per_node)
directors = jax.vmap(_utils.rotate_directors)(
  jnp.asarray(u_nodal[:, 3:]), rod.directors_0
)
kinematics = _mesher.compute_discrete_cosserat_kinematics(
  node_coords=rod.node_coords + jnp.asarray(u_nodal[:, :3]),
  directors=directors,
  arc_coords=rod.arc_coords,
)
nu_num = float(np.mean(np.asarray(kinematics.strain)[:, 2]))
kappa_num = float(np.mean(np.asarray(kinematics.curvature)[:, 1]))

print(
  f"nu      : exact {nu_exact:.6f}   crops {nu_num:.6f}   "
  f"err {abs(nu_num-nu_exact)/nu_exact:.3%}"
)
print(
  f"kappa_2 : exact {curvature_exact:.6f}   crops {kappa_num:.6f}   "
  f"err {abs(kappa_num-curvature_exact)/abs(curvature_exact):.3%}"
)

In [ ]:
deformed = np.asarray(rod.node_coords) + u_nodal[:, :3]
angles = curvature_exact * np.asarray(rod.arc_coords)
arc_x = arc_radius * np.sin(angles)
arc_y = arc_radius * (1.0 - np.cos(angles) )

fig, (ax_shape, ax_strain) = plt.subplots(1, 2, figsize=(11, 4.5))
ax_shape.plot(arc_x, -arc_y, "k-", lw=2, label="exact circular arc")
ax_shape.plot(deformed[:, 0], deformed[:, 1], "o", ms=3, mfc="none", label="CROPS")
ax_shape.set_aspect("equal")
ax_shape.set_xlabel("$x$ [m]")
ax_shape.set_ylabel("$y$ [m]")
ax_shape.legend()

ax_strain.plot(rod.elem_centers[:, 0], kinematics.strain[:, 2], label=r"$\nu$")
ax_strain.axhline(nu_exact, color="k", ls="--", lw=1)
ax_strain.set_xlabel("reference $x$ [m]")
ax_strain.legend()
fig.tight_layout()
plt.show()

In [ ]:
history = np.asarray(u_history).reshape(num_steps, num_nodes, -1)

fig, ax = plt.subplots(figsize=(5.5, 5))
for step in range(0, num_steps, max(1, num_steps // 6)):
  shape = np.asarray(rod.node_coords) + history[step, :, :3]
  ax.plot(shape[:, 0], shape[:, 1], lw=1, label=rf"$\mu$ = {chem_potentials[step]:.0f}")
ax.set_aspect("equal")
ax.set_xlabel("$x$ [m]")
ax.set_ylabel("$y$ [m]")
ax.legend(fontsize=7)
plt.show()

### Grad check

In [ ]:
def objective(flory_parameters):
  """Scalar objective: tip y-displacement of the swelling bilayer."""
  u_star, _ = problem.solve(
    u0=jnp.zeros(rod.num_dofs),
    shear_moduli=jnp.full((num_elems, num_fibers), shear_mod),
    flory_parameters=flory_parameters,
    chem_potentials=chem_potentials,
  )
  return u_star.reshape(num_nodes, rod.num_dofs_per_node)[-1, 1]


grad_reverse = jax.grad(objective)(flory_parameters)

# Central differences on a handful of entries. Use a tight Newton tolerance,
# otherwise the solver noise floor swamps the finite difference.
step = 1e-6
probe = [(0, 0), (num_elems // 2, 0), (num_elems // 2, 1), (num_elems - 1, 1)]
print(f"{'entry':>12} {'jax.grad':>16} {'central FD':>16} {'rel err':>12}")
for idx in probe:
  plus = flory_parameters.at[idx].add(step)
  minus = flory_parameters.at[idx].add(-step)
  fd = (objective(plus) - objective(minus)) / (2.0 * step)
  ad = grad_reverse[idx]
  print(f"{str(idx):>12} {ad:16.8e} {fd:16.8e} "
        f"{abs(ad - fd) / max(abs(fd), 1e-30):12.2e}")